# 🎯 Sentiment Analysis — Komentar TikTok Koperasi Desa Merah Putih (KDMP)

**Dataset:** sinryurifal/dataset-komentar-tiktok-koperasi-desa-merah-putih  
**Tujuan:** Menganalisis sentimen publik terhadap program KDMP dari komentar TikTok  
**Metode:** IndoBERT / Lexicon-based + Visualisasi

---

## 📦 1. Install Library

In [ ]:
!pip install kagglehub pandas numpy matplotlib seaborn wordcloud
!pip install transformers torch scikit-learn
!pip install Sastrawi  # stemmer Bahasa Indonesia
!pip install nlp-id   # NLP tools Bahasa Indonesia

## 🔑 2. Setup Kaggle API Token

Masukkan token Kaggle kamu. Buka **kaggle.com → Settings → API → Create New Token**

In [ ]:
import os
import json

# ============================================================
# ⚠️  GANTI dengan username dan API key Kaggle kamu!
# ============================================================
KAGGLE_USERNAME = "sinryurifal"       # ganti username kamu
KAGGLE_KEY      = "KGAT_14a057d7b6c0d67d2430cacf55f1320e"  # ganti key kamu
# ============================================================

# Buat folder ~/.kaggle dan simpan kredensial
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
kaggle_json = {"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}

with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    json.dump(kaggle_json, f)

os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
print("✅ Kaggle credentials saved!")

## ⬇️ 3. Download & Extract Dataset dari Kaggle

In [ ]:
import kagglehub
import zipfile
import glob

# Download dataset (otomatis unzip)
path = kagglehub.dataset_download("sinryurifal/dataset-komentar-tiktok-koperasi-desa-merah-putih")
print(f"📁 Dataset tersimpan di: {path}")

# List semua file
all_files = glob.glob(f"{path}/**/*", recursive=True)
print("\n📄 File yang tersedia:")
for f in all_files:
    print(" ", f)

In [ ]:
import pandas as pd

# Cari file CSV secara otomatis
csv_files = glob.glob(f"{path}/**/*.csv", recursive=True)
print(f"CSV ditemukan: {csv_files}")

# Load semua CSV (gabungkan jika lebih dari satu)
dfs = []
for csv_file in csv_files:
    print(f"Loading: {csv_file}")
    df_temp = pd.read_csv(csv_file, encoding='utf-8', on_bad_lines='skip')
    dfs.append(df_temp)
    print(f"  Shape: {df_temp.shape}")
    print(f"  Columns: {df_temp.columns.tolist()}")

df = pd.concat(dfs, ignore_index=True) if len(dfs) > 1 else dfs[0]
print(f"\n✅ Total data: {df.shape}")
df.head()

## 🔍 4. Eksplorasi Data (EDA)

In [ ]:
# Info dasar dataset
print("=" * 50)
print("INFO DATASET")
print("=" * 50)
print(f"Jumlah baris   : {len(df):,}")
print(f"Jumlah kolom   : {len(df.columns)}")
print(f"Kolom          : {df.columns.tolist()}")
print("\nMissing values:")
print(df.isnull().sum())
print("\nContoh data:")
df.head(10)

In [ ]:
# ============================================================
# Sesuaikan nama kolom teks komentar di bawah ini
# Kemungkinan: 'comment', 'text', 'komentar', 'Comment', dll.
# ============================================================

# Deteksi otomatis kolom teks
text_candidates = [c for c in df.columns if any(k in c.lower() for k in ['comment','text','komentar','isi','content'])]
print(f"Kandidat kolom teks: {text_candidates}")

# Set kolom teks (ganti jika perlu)
TEXT_COL = text_candidates[0] if text_candidates else df.columns[0]
print(f"\n✅ Menggunakan kolom: '{TEXT_COL}'")
print(df[TEXT_COL].head(10))

## 🧹 5. Preprocessing Teks

In [ ]:
import re
import string
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Inisialisasi stopword & stemmer Bahasa Indonesia
stop_factory = StopWordRemoverFactory()
stopwords_id = stop_factory.get_stop_words()

stem_factory = StemmerFactory()
stemmer = stem_factory.create_stemmer()

def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = text.lower()                                     # lowercase
    text = re.sub(r'@\w+', '', text)                        # hapus mention
    text = re.sub(r'#\w+', '', text)                        # hapus hashtag
    text = re.sub(r'http\S+|www\.\S+', '', text)           # hapus URL
    text = re.sub(r'[^\w\s]', ' ', text)                   # hapus tanda baca
    text = re.sub(r'\d+', '', text)                         # hapus angka
    text = re.sub(r'\s+', ' ', text).strip()               # normalisasi spasi
    # Hapus emoji (unicode)
    text = text.encode('ascii', 'ignore').decode('ascii')
    # Hapus stopwords
    words = [w for w in text.split() if w not in stopwords_id and len(w) > 2]
    text = ' '.join(words)
    return text

print("🔄 Preprocessing...")
df['text_clean'] = df[TEXT_COL].apply(preprocess_text)
print("✅ Selesai!")

# Tampilkan perbandingan
comparison = pd.DataFrame({
    'Original': df[TEXT_COL].head(5).values,
    'Clean'   : df['text_clean'].head(5).values
})
print(comparison.to_string())

In [ ]:
# Hapus baris kosong setelah preprocessing
df_clean = df[df['text_clean'].str.strip() != ''].copy()
print(f"Data sebelum cleaning: {len(df):,}")
print(f"Data setelah cleaning : {len(df_clean):,}")

## 🤖 6. Sentiment Analysis

### Metode A: IndoBERT (Deep Learning — lebih akurat)

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch

print("📥 Loading IndoBERT sentiment model...")
MODEL_NAME = "mdhugol/indonesia-bert-sentiment-classification"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

sentiment_pipeline = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)
print("✅ Model loaded! GPU:", torch.cuda.is_available())

In [ ]:
from tqdm import tqdm

# Label mapping
label_map = {'LABEL_0': 'Positif', 'LABEL_1': 'Netral', 'LABEL_2': 'Negatif'}

def predict_sentiment_batch(texts, batch_size=32):
    results = []
    # Truncate teks agar tidak melebihi max token
    texts_trunc = [t[:512] for t in texts]
    for i in tqdm(range(0, len(texts_trunc), batch_size), desc="Predicting"):
        batch = texts_trunc[i:i+batch_size]
        batch = [t if t.strip() else "netral" for t in batch]  # hindari teks kosong
        preds = sentiment_pipeline(batch, truncation=True, max_length=512)
        results.extend(preds)
    return results

print("🔄 Menjalankan prediksi sentimen (mungkin beberapa menit)...")
predictions = predict_sentiment_batch(df_clean['text_clean'].tolist())

df_clean['sentiment_raw']   = [p['label'] for p in predictions]
df_clean['sentiment']       = df_clean['sentiment_raw'].map(label_map)
df_clean['sentiment_score'] = [round(p['score'], 4) for p in predictions]

print("\n✅ Prediksi selesai!")
print(df_clean[['text_clean', 'sentiment', 'sentiment_score']].head(10))

## 📊 7. Visualisasi Hasil

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 12

COLORS = {'Positif': '#2ecc71', 'Netral': '#3498db', 'Negatif': '#e74c3c'}

sentiment_counts = df_clean['sentiment'].value_counts()
sentiment_pct    = (sentiment_counts / len(df_clean) * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Analisis Sentimen Komentar TikTok\nKoperasi Desa Merah Putih (KDMP)', 
             fontsize=15, fontweight='bold', y=1.02)

# --- Bar chart ---
ax1 = axes[0]
bars = ax1.bar(sentiment_counts.index, sentiment_counts.values,
               color=[COLORS.get(s, '#95a5a6') for s in sentiment_counts.index],
               edgecolor='white', linewidth=1.5, width=0.5)
for bar, val, pct in zip(bars, sentiment_counts.values, sentiment_pct.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             f'{val:,}\n({pct}%)', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax1.set_title('Distribusi Sentimen', fontsize=13, fontweight='bold')
ax1.set_xlabel('Sentimen')
ax1.set_ylabel('Jumlah Komentar')
ax1.set_ylim(0, max(sentiment_counts.values) * 1.2)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.grid(axis='y', alpha=0.3)

# --- Pie chart ---
ax2 = axes[1]
wedge_colors = [COLORS.get(s, '#95a5a6') for s in sentiment_counts.index]
wedges, texts, autotexts = ax2.pie(
    sentiment_counts.values,
    labels=sentiment_counts.index,
    autopct='%1.1f%%',
    colors=wedge_colors,
    startangle=90,
    explode=[0.05]*len(sentiment_counts),
    textprops={'fontsize': 12}
)
for autotext in autotexts:
    autotext.set_fontweight('bold')
ax2.set_title('Proporsi Sentimen', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('sentiment_distribution.png', bbox_inches='tight')
plt.show()
print(sentiment_pct)

In [ ]:
from wordcloud import WordCloud

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Word Cloud per Kategori Sentimen', fontsize=15, fontweight='bold')

for idx, (sent, color) in enumerate(COLORS.items()):
    texts_sent = df_clean[df_clean['sentiment'] == sent]['text_clean'].dropna()
    all_words = ' '.join(texts_sent.tolist())
    
    if all_words.strip():
        wc = WordCloud(
            width=600, height=400,
            background_color='white',
            colormap='Greens' if sent=='Positif' else ('Blues' if sent=='Netral' else 'Reds'),
            max_words=100,
            collocations=False
        ).generate(all_words)
        axes[idx].imshow(wc, interpolation='bilinear')
    
    axes[idx].set_title(f'Sentimen {sent} ({len(texts_sent):,})', 
                        fontsize=13, fontweight='bold', color=color)
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('wordcloud_sentiment.png', bbox_inches='tight')
plt.show()

In [ ]:
# Distribusi confidence score
fig, ax = plt.subplots(figsize=(10, 5))
for sent, color in COLORS.items():
    data = df_clean[df_clean['sentiment'] == sent]['sentiment_score']
    ax.hist(data, bins=30, alpha=0.6, label=sent, color=color, edgecolor='white')

ax.set_title('Distribusi Confidence Score per Sentimen', fontsize=13, fontweight='bold')
ax.set_xlabel('Confidence Score')
ax.set_ylabel('Frekuensi')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('confidence_distribution.png', bbox_inches='tight')
plt.show()

## 📋 8. Contoh Komentar per Sentimen

In [ ]:
for sent in ['Positif', 'Negatif', 'Netral']:
    print(f"\n{'='*60}")
    print(f"🔹 Contoh Komentar {sent.upper()} (confidence > 0.85):")
    print('='*60)
    
    sample = df_clean[
        (df_clean['sentiment'] == sent) & 
        (df_clean['sentiment_score'] > 0.85)
    ][TEXT_COL].dropna().sample(min(5, len(df_clean[df_clean['sentiment']==sent])), random_state=42)
    
    for i, text in enumerate(sample.values, 1):
        print(f"{i}. {text[:200]}")

## 💾 9. Simpan Hasil

In [ ]:
# Simpan hasil ke CSV
output_cols = [TEXT_COL, 'text_clean', 'sentiment', 'sentiment_score']
output_cols = [c for c in output_cols if c in df_clean.columns]

df_result = df_clean[output_cols].copy()
df_result.to_csv('hasil_sentimen_kdmp.csv', index=False, encoding='utf-8-sig')
print("✅ Hasil disimpan ke: hasil_sentimen_kdmp.csv")

# Ringkasan statistik
print("\n📊 RINGKASAN ANALISIS SENTIMEN")
print("=" * 40)
print(f"Total komentar dianalisis: {len(df_clean):,}")
print()
for sent in ['Positif', 'Netral', 'Negatif']:
    count = (df_clean['sentiment'] == sent).sum()
    pct   = count / len(df_clean) * 100
    avg_score = df_clean[df_clean['sentiment']==sent]['sentiment_score'].mean()
    print(f"  {sent:10s}: {count:5,} ({pct:5.1f}%)  avg confidence: {avg_score:.3f}")

In [ ]:
# Download hasil ke lokal (khusus Google Colab)
try:
    from google.colab import files
    files.download('hasil_sentimen_kdmp.csv')
    files.download('sentiment_distribution.png')
    files.download('wordcloud_sentiment.png')
    print("✅ File berhasil di-download!")
except ImportError:
    print("ℹ️ Bukan di Colab — file sudah tersimpan di direktori saat ini.")

---
## ✅ Selesai!

**Apa yang sudah dilakukan:**
1. ✅ Download dataset otomatis dari Kaggle  
2. ✅ Preprocessing (hapus mention, hashtag, emoji, stopwords)  
3. ✅ Prediksi sentimen dengan IndoBERT (model khusus Bahasa Indonesia)  
4. ✅ Visualisasi distribusi sentimen & word cloud  
5. ✅ Export hasil ke CSV  

**Model yang digunakan:** `mdhugol/indonesia-bert-sentiment-classification`  
Label: **Positif**, **Netral**, **Negatif**